# PR11: Advanced Machine Learning and Model Optimization (CO5)

**Aim:** Implement and compare Support Vector Machine (SVM), Random Forest, and AdaBoost classifiers. Apply cross-validation and Grid Search for hyperparameter tuning, then compare performance before and after optimization.

**Dataset:** Breast Cancer Wisconsin dataset from Scikit-learn.

## 1. Objectives

1. Implement SVM, Random Forest, and AdaBoost classifiers.
2. Evaluate models using cross-validation.
3. Optimize hyperparameters using Grid Search.
4. Compare model performance before and after optimization.

## 2. Theory

- **Support Vector Machine (SVM):** Finds a decision boundary that separates classes with a maximum margin.
- **Random Forest:** Combines multiple decision trees and aggregates their predictions.
- **AdaBoost:** Combines weak learners sequentially, emphasizing samples that previous learners misclassified.
- **Cross-validation:** Splits training data into multiple folds to estimate model performance more reliably.
- **Grid Search:** Tests combinations of specified hyperparameters using cross-validation and selects the best combination.

**Evaluation metrics:** Accuracy, precision, recall, F1-score, and ROC-AUC.

## 3. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

## 4. Load the Dataset

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target

print("Dataset:", data.DESCR.splitlines()[0])
print("Number of samples:", X.shape[0])
print("Number of features:", X.shape[1])
print("Classes:", data.target_names)
print("Class counts:", dict(zip(*np.unique(y, return_counts=True))))

## 5. Split Data into Training and Testing Sets

In [ ]:
# Keep the test set separate until final evaluation
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 6. Define the Models Before Optimization

SVM is placed in a pipeline with StandardScaler because it is sensitive to feature scale. Random Forest and AdaBoost are used with their default settings as baseline models.

In [ ]:
baseline_models = {
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(probability=True, random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42)
}

print("Baseline models defined:", ", ".join(baseline_models.keys()))

## 7. Evaluate Baseline Models Using 5-Fold Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

baseline_cv_rows = []

for name, model in baseline_models.items():
    scores = cross_validate(
        model, X_train, y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )
    baseline_cv_rows.append({
        "Model": name,
        "CV Accuracy (%)": scores["test_accuracy"].mean() * 100,
        "CV Precision (%)": scores["test_precision"].mean() * 100,
        "CV Recall (%)": scores["test_recall"].mean() * 100,
        "CV F1 (%)": scores["test_f1"].mean() * 100,
        "CV ROC-AUC (%)": scores["test_roc_auc"].mean() * 100
    })

baseline_cv_df = pd.DataFrame(baseline_cv_rows).round(2)
baseline_cv_df

## 8. Define Hyperparameter Grids

Grid Search evaluates each specified parameter combination using the same 5-fold stratified cross-validation. The grids are intentionally moderate so they are suitable for a practical lab exercise.

In [ ]:
param_grids = {
    "SVM": {
        "model__C": [0.1, 1, 10],
        "model__kernel": ["linear", "rbf"],
        "model__gamma": ["scale", "auto"]
    },
    "Random Forest": {
        "n_estimators": [100, 200],
        "max_depth": [None, 10, 20],
        "min_samples_split": [2, 5]
    },
    "AdaBoost": {
        "n_estimators": [50, 100, 200],
        "learning_rate": [0.01, 0.1, 1.0]
    }
}

print("Hyperparameter grids prepared for all three models.")

## 9. Perform Grid Search Hyperparameter Tuning

In [ ]:
optimized_models = {}
grid_searches = {}
optimization_rows = []

for name, model in baseline_models.items():
    print(f"Running Grid Search for {name}...")

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grids[name],
        scoring="accuracy",
        cv=cv,
        n_jobs=-1,
        refit=True
    )
    search.fit(X_train, y_train)

    grid_searches[name] = search
    optimized_models[name] = search.best_estimator_

    optimization_rows.append({
        "Model": name,
        "Best CV Accuracy (%)": search.best_score_ * 100,
        "Best Parameters": search.best_params_
    })

optimization_df = pd.DataFrame(optimization_rows)
optimization_df

## 10. Evaluate Baseline and Optimized Models on the Held-Out Test Set

In [ ]:
def evaluate_classifier(model, X_eval, y_eval):
    y_pred = model.predict(X_eval)

    # Use probability scores when available; otherwise use decision scores
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_eval)[:, 1]
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_eval)
    else:
        y_score = y_pred

    return {
        "Accuracy (%)": accuracy_score(y_eval, y_pred) * 100,
        "Precision (%)": precision_score(y_eval, y_pred, zero_division=0) * 100,
        "Recall (%)": recall_score(y_eval, y_pred, zero_division=0) * 100,
        "F1-score (%)": f1_score(y_eval, y_pred, zero_division=0) * 100,
        "ROC-AUC (%)": roc_auc_score(y_eval, y_score) * 100
    }

test_rows = []

for name, model in baseline_models.items():
    metrics = evaluate_classifier(model.fit(X_train, y_train), X_test, y_test)
    test_rows.append({"Model": name, "Stage": "Before optimization", **metrics})

for name, model in optimized_models.items():
    metrics = evaluate_classifier(model, X_test, y_test)
    test_rows.append({"Model": name, "Stage": "After optimization", **metrics})

test_results_df = pd.DataFrame(test_rows).round(2)
test_results_df

## 11. Compare Accuracy Before and After Optimization

In [ ]:
accuracy_comparison = test_results_df.pivot(
    index="Model", columns="Stage", values="Accuracy (%)"
)
accuracy_comparison["Change (percentage points)"] = (
    accuracy_comparison["After optimization"]
    - accuracy_comparison["Before optimization"]
)
accuracy_comparison.round(2)

## 12. Visualize Test Accuracy

In [ ]:
plot_df = test_results_df.pivot(
    index="Model", columns="Stage", values="Accuracy (%)"
)

ax = plot_df.plot(kind="bar", figsize=(9, 5))
ax.set_title("Model Accuracy Before and After Optimization")
ax.set_ylabel("Test Accuracy (%)")
ax.set_xlabel("Classifier")
ax.set_ylim(0, 105)
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.legend(title="Stage")
plt.tight_layout()
plt.show()

## 13. Best Model and Parameters

The best model is selected by Grid Search's mean cross-validation accuracy. This is a training-stage selection criterion; the held-out test metrics above are used for final comparison.

In [ ]:
best_model_name = max(
    grid_searches,
    key=lambda name: grid_searches[name].best_score_
)
best_search = grid_searches[best_model_name]

print("Best model based on cross-validation:", best_model_name)
print(f"Best mean CV accuracy: {best_search.best_score_ * 100:.2f}%")
print("Best hyperparameters:", best_search.best_params_)

## 14. Result

SVM, Random Forest, and AdaBoost were implemented and evaluated using stratified 5-fold cross-validation. Grid Search was used to tune their hyperparameters. The notebook reports cross-validation metrics, the best parameters for each model, and held-out test performance before and after optimization.

## 15. Conclusion

Cross-validation provides a more reliable estimate of model performance during training, while Grid Search helps identify promising hyperparameter settings. Comparing held-out test metrics shows whether optimization improved performance. Results may differ slightly across software versions; a tuned model is not guaranteed to outperform its baseline on every test metric.